In [2]:
import sys
import os
sys.path.append(os.path.abspath('../utils'))

import pandas as pd
import numpy as np
from data_aggregation_tools import merge_tickets_assets

df_tickets = pd.read_csv("../../data/V_OM_WORK_TASK.csv")
df_assets = pd.read_csv("../../data/V_OM_WORK_TASK_ASSET.csv")

# Generate merged tickets / assets file
df_tickets_assets = merge_tickets_assets(df_tickets=df_tickets, df_assets=df_assets)

print(f"shape of df_tickets: {df_tickets.shape}")
print(f"shape of df_assets: {df_assets.shape}")
print(f"shape of df_tickets_assets: {df_tickets_assets.shape}")

/var/folders/y2/tsyh3wpj3mj7zcv4hfd0knl40000gn/T/ipykernel_66106/253192481.py:9: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_tickets = pd.read_csv("../../data/V_OM_WORK_TASK.csv")
/var/folders/y2/tsyh3wpj3mj7zcv4hfd0knl40000gn/T/ipykernel_66106/253192481.py:10: DtypeWarning: Columns (0,3,4,5,6,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_assets = pd.read_csv("../../data/V_OM_WORK_TASK_ASSET.csv")


shape of df_tickets: (227205, 47)
shape of df_assets: (316257, 9)
shape of df_tickets_assets: (316257, 55)


In [4]:
print(df_tickets_assets['BASELINE_START_LTZ'].dtype)

object


In [3]:
# Sort the DataFrame by ASSET_ID and BASELINE_START_LTZ for efficient processing
df_merged_assets_sorted = df_tickets_assets.sort_values(by=['ASSET_ID', 'BASELINE_START_LTZ']).reset_index(drop=True)

repeated_tasks_by_asset_list = []
num_days = pd.Timedelta(days=90) 

print(num_days)

# Group by ASSET_ID
for asset_id, group in df_merged_assets_sorted.groupby('ASSET_ID'):
    # Iterate through each task in the group
    for i in range(len(group)):
        current_task = group.iloc[i]
        current_task_time = current_task['BASELINE_START_LTZ']

        # Check subsequent tasks for repetition within 90 days
        # Start checking from the next task (i+1) within the same asset group
        subsequent_tasks = group.iloc[i+1:]
        for j in range(len(subsequent_tasks)):
            other_task = subsequent_tasks.iloc[j]
            other_task_time = other_task['BASELINE_START_LTZ']

            time_difference = other_task_time - current_task_time

            if pd.isna(time_difference): # Skip if time_difference is NaT
                continue

            if time_difference <= num_days:
                # If a subsequent task is found within 90 days, the current task is repetitive
                repeated_tasks_by_asset_list.append(current_task.to_dict())
                break  # Move to the next current_task, as this one is identified as repetitive
            elif time_difference > num_days:
                # Since tasks are sorted by time, if the current subsequent task is already
                # beyond 90 days, all further tasks will also be, so we can break early.
                break

# Create a new DataFrame from the identified repeated tasks
df_tickets_filtered_by_asset = pd.DataFrame(repeated_tasks_by_asset_list)

print(f"Found {len(df_tickets_filtered_by_asset)} repetitive tasks by asset within {num_days.days} days.")



90 days 00:00:00


TypeError: unsupported operand type(s) for -: 'str' and 'str'

# Create Chart of Assets that are repeated

Here, we check for assets that have been repaired for corrective work more than once in a span of 90 days.

The 'Number of Recurrences' is the number of these 90-day intervals present for that specific asset in the data.

In [ ]:
import plotly.express as px
import pandas as pd

asset_recurrence_counts = df_tickets_filtered_by_asset['ASSET_NAME'].value_counts()

# Create a temporary DataFrame from the asset_recurrence_counts Series
df_plot_asset_recurrence = pd.DataFrame({
    'ASSET_NAME': asset_recurrence_counts.index,
    'Recurrence Count': asset_recurrence_counts.values
})

# Get a mapping of ASSET_NAME to ASSET_PRIMARY_LOCATION from the original filtered data
# Since an ASSET_NAME might appear multiple times but always with the same primary location
# (or we want to show a representative one), we'll take the first unique location.
asset_location_mapping = df_tickets_filtered_by_asset.groupby('ASSET_NAME')['ASSET_PRIMARY_LOCATION'].apply(lambda x: ', '.join(x.dropna().astype(str).unique())).reset_index()

# Merge the location information into the plot DataFrame
df_plot_asset_recurrence = pd.merge(df_plot_asset_recurrence, asset_location_mapping, on='ASSET_NAME', how='left')

# Create an interactive bar chart using Plotly Express
fig = px.bar(
    df_plot_asset_recurrence,
    x='ASSET_NAME',
    y='Recurrence Count',
    title='Recurrence Counts of Repetitive Tasks by Asset Name (Interactive)',
    color='Recurrence Count',
    color_continuous_scale=px.colors.sequential.Viridis,
    labels={'ASSET_NAME': 'Asset Name', 'Recurrence Count': 'Number of Recurrences', 'ASSET_PRIMARY_LOCATION': 'Primary Location'},
    hover_data=['ASSET_PRIMARY_LOCATION'] 
)

# Update layout for better readability
fig.update_layout(
    xaxis_tickangle=-90,
    xaxis_title_standoff=25,
    height=700
)

# Save the interactive chart as an HTML file
output_html_filename = 'repetitive_tasks_by_asset_interactive_new.html'
fig.write_html(output_html_filename)

print(f"Interactive chart saved as '{output_html_filename}'")

Interactive chart saved as 'repetitive_tasks_by_asset_interactive_new.html'
